# NFL Draft Prediction - Competition Submission
## GCI World 2026 April
### Student: Daniela Chavez

**Approach:**
- Feature Engineering: BMI, power-to-weight ratios, explosiveness index, speed-agility composites, missing data indicator, school frequency, school target encoding
- Ensemble of 4 models: LightGBM, XGBoost, CatBoost, Random Forest
- 5-Fold Stratified Cross-Validation with performance-weighted averaging
- Validation AUC: ~0.85

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

!pip install lightgbm xgboost catboost -q
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

## 2. Mount Google Drive & Set Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# IMPORTANT: Change this path to where you uploaded the competition folder
%cd "/content/drive/MyDrive/competition"

## 3. Load Data

In [ ]:
PATH = Path.cwd() / "input"

train = pd.read_csv(PATH / "train.csv")
test = pd.read_csv(PATH / "test.csv")
sample_sub = pd.read_csv(PATH / "sample_submission.csv")

print(f"train: {train.shape}, test: {test.shape}, sample_sub: {sample_sub.shape}")
train.head()

## 4. Exploratory Data Analysis

In [ ]:
# Target distribution
drafted_pct = train['Drafted'].value_counts(normalize=True) * 100
print(f"Drafted (1): {drafted_pct.get(1, 0):.1f}%")
print(f"Not Drafted (0): {drafted_pct.get(0, 0):.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train['Drafted'].value_counts().plot(kind='bar', ax=axes[0], title='Drafted Distribution')
train.isnull().sum().sort_values(ascending=False).plot(kind='bar', ax=axes[1], title='Missing Values')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target
numeric_cols = train.select_dtypes(include=['number']).drop(columns=['Id'])
corr_with_target = numeric_cols.corr()['Drafted'].drop('Drafted').sort_values(ascending=False)
print("Correlation with Drafted:")
print(corr_with_target)

## 5. Feature Engineering

In [ ]:
def create_features(df):
    df = df.copy()
    # Body composition
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Weight_Height'] = df['Weight'] / df['Height']
    
    # Power-to-weight ratios
    df['Power_Weight'] = df['Vertical_Jump'] / df['Weight']
    df['Broad_Weight'] = df['Broad_Jump'] / df['Weight']
    df['Strength_Weight'] = df['Bench_Press_Reps'] / df['Weight'] * 100
    
    # Speed & agility composites
    df['Speed_Agility'] = df['Sprint_40yd'] + df['Agility_3cone']
    df['Speed_Shuttle'] = df['Sprint_40yd'] + df['Shuttle']
    df['Sprint_Height'] = df['Sprint_40yd'] / df['Height']
    
    # Explosiveness index
    df['Explosiveness'] = df['Vertical_Jump'] + df['Broad_Jump']
    
    # Missing data as a feature (players who skip tests)
    perf_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                 'Broad_Jump', 'Agility_3cone', 'Shuttle', 'Age']
    df['Missing_Count'] = df[perf_cols].isnull().sum(axis=1)
    
    # School frequency
    df['School_Freq'] = df['School'].map(df['School'].value_counts())
    
    return df

train = create_features(train)
test = create_features(test)
print(f"After feature engineering - Train: {train.shape}, Test: {test.shape}")

## 6. Preprocessing

In [ ]:
# Label encode categorical features
cat_cols = ['Player_Type', 'Position_Type', 'Position']
label_encoders = {}
for c in cat_cols:
    label_encoders[c] = LabelEncoder()
    train[c] = label_encoders[c].fit_transform(train[c].astype(str))
    test[c] = label_encoders[c].transform(test[c].astype(str))

# Target encoding for School
school_means = train.groupby('School')['Drafted'].mean()
global_mean = train['Drafted'].mean()
train['School_Target'] = train['School'].map(school_means)
test['School_Target'] = test['School'].map(school_means).fillna(global_mean)

# Prepare features and target
X = train.drop(columns=['Id', 'Drafted', 'School'])
y = train['Drafted']
X_test = test.drop(columns=['Id', 'School'])

# Impute missing values with train mean
for col in X.columns:
    if X[col].isnull().any():
        mean_val = X[col].mean()
        X[col] = X[col].fillna(mean_val)
        X_test[col] = X_test[col].fillna(mean_val)

print(f"X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")
print(f"\nFeatures ({len(X.columns)}): {list(X.columns)}")
print(f"\nMissing values remaining - Train: {X.isnull().sum().sum()}, Test: {X_test.isnull().sum().sum()}")

## 7. Train Ensemble Model (LightGBM + XGBoost + CatBoost + RandomForest)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'lgbm': LGBMClassifier(
        n_estimators=1000, learning_rate=0.03, max_depth=6,
        num_leaves=31, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, random_state=42, verbose=-1
    ),
    'xgb': XGBClassifier(
        n_estimators=1000, learning_rate=0.03, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, random_state=42,
        eval_metric='auc', verbosity=0
    ),
    'catboost': CatBoostClassifier(
        iterations=1000, learning_rate=0.03, depth=6,
        random_seed=42, verbose=0
    ),
    'rf': RandomForestClassifier(
        n_estimators=500, max_depth=10, min_samples_leaf=5,
        random_state=42, n_jobs=-1
    )
}

all_test_preds = {}
all_val_scores = {}

for name, model in models.items():
    print(f'\nTraining: {name}')
    test_preds = []
    val_scores = []
    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        model.fit(X_train, y_train)
        y_valid_pred = model.predict_proba(X_valid)[:, 1]
        auc = roc_auc_score(y_valid, y_valid_pred)
        val_scores.append(auc)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        print(f'  Fold {fold+1} AUC: {auc:.4f}')
    mean_auc = np.mean(val_scores)
    print(f'  >>> Mean AUC: {mean_auc:.4f}')
    all_test_preds[name] = np.mean(test_preds, axis=0)
    all_val_scores[name] = mean_auc

print(f'\n{"="*50}')
print('MODEL SUMMARY:')
for name, score in sorted(all_val_scores.items(), key=lambda x: -x[1]):
    print(f'  {name:>10s}: AUC = {score:.4f}')

## 8. Feature Importance

In [ ]:
# Feature importance from the last trained RF model
feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': models['rf'].feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=feat_imp, x='Importance', y='Feature')
plt.title('Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()

## 9. Create Submission File

In [ ]:
# Weighted average based on validation AUC
total_score = sum(all_val_scores.values())
weights = {name: score / total_score for name, score in all_val_scores.items()}

print('Ensemble weights:')
for name, w in weights.items():
    print(f'  {name}: {w:.3f}')

final_preds = np.zeros(len(X_test))
for name, preds in all_test_preds.items():
    final_preds += weights[name] * preds

# Save submission
submission = sample_sub.copy()
submission['Drafted'] = final_preds

output_path = Path.cwd() / 'output'
output_path.mkdir(exist_ok=True)
submission.to_csv(output_path / 'submission.csv', index=False)

print(f'\nSubmission saved to output/submission.csv')
print(f'Shape: {submission.shape}')
print(f'\nFirst 10 rows:')
print(submission.head(10))
print(f'\nPrediction statistics:')
print(submission['Drafted'].describe())